# optimizer-init-params-list — ex1: materialize a generator of params into a list at init

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-init-params-list`. Running the final beacon cell reports progress against the `PyTorch: Optimizer init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Optimizer init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-init-params-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-init-params-list"
DD_SUBTOPIC = "PyTorch: Optimizer init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `optim.SGD(params, lr=...)` — quick refresher

An optimizer is constructed with an iterable of `nn.Parameter` tensors. Internally PyTorch's optimizer immediately materializes that iterable into a list — but if you ROLL YOUR OWN optimizer (as the ARENA SGD exercise does), you must do it yourself: `self.params = list(params)`. The reason: a generator can only be iterated once. If you store the generator and iterate it during `.step()`, the second call hits an empty iterator and silently does nothing.

**The model.parameters() trap.** `model.parameters()` returns a generator. If your optimizer stores `self.params = params` instead of `list(params)`, the first `.step()` consumes it and every subsequent step is a no-op. Tests pass on iteration 1 and fail mysteriously on iteration 2.

### Exercise 1 — materialize a generator of params into a list at init

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze why `self.params = list(params)` is required in a hand-rolled optimizer's `__init__` and implement the fix so that the optimizer survives being passed a generator.
> Keywords: optimizer-init, generator, list-materialization
> ```

**KCs targeted:** `optimizer-init-list-vs-generator`, `optimizer-init-stores-params-attribute`

You are given a `BuggyOptimizer` whose `__init__` stores the raw `params` iterable (often a generator). On the second call to `.step()` the optimizer silently does nothing because the generator was already consumed.

Implement `Ex1FixedOptimizer.__init__(self, params, lr)` to fix the bug. The contract:

1. Materialize the iterable: `self.params = list(params)`.
2. Store `self.lr = lr`.
3. Provide a `.step()` method that does an in-place vanilla SGD update for every param that has a non-None `.grad`:
   `p.data -= self.lr * p.grad`.
4. Provide a `.zero_grad()` method that sets every `p.grad = None`.

The test verifies:
- The fixed optimizer works when given a `model.parameters()` generator (the canonical PyTorch pattern).
- The fixed optimizer's `.step()` actually mutates params on the SECOND call (the buggy one fails this).
- The buggy version is demonstrably broken on the second `.step()` for the same input.

Decorate `.step()` with `@t.no_grad()` so the in-place mutation doesn't pollute the autograd graph.

In [ ]:
class BuggyOptimizer:
    # BUG: stores generator as-is
    def __init__(self, params, lr):
        self.params = params   # <- never materialized to a list
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:           # generator: empty on 2nd call
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex1FixedOptimizer:
    """Optimizer that materializes `params` into a list at init."""

    def __init__(self, params, lr):
        raise NotImplementedError()

    @t.no_grad()
    def step(self):
        raise NotImplementedError()

    def zero_grad(self):
        raise NotImplementedError()


def _test_ex1():
    # Build a tiny model and pass its PARAMETER GENERATOR to the optimizers.
    model = t.nn.Linear(3, 1, bias=False)
    with t.no_grad():
        model.weight.copy_(t.zeros_like(model.weight))

    # Sanity: model.parameters() is indeed a generator.
    import types
    params_gen = model.parameters()
    assert isinstance(params_gen, types.GeneratorType), (
        f'model.parameters() should be a generator; got {type(params_gen)}'
        f'  (PyTorch version mismatch?)'
    )

    fixed = Ex1FixedOptimizer(model.parameters(), lr=0.1)
    # After construction, the FIXED optimizer must store a LIST.
    assert isinstance(fixed.params, list), (
        f'fixed.params must be a list, got {type(fixed.params)}; '
        f'did you forget list(params)?'
    )
    assert len(fixed.params) == 1, f'expected 1 param, got {len(fixed.params)}'

    # Do two real training steps; weight should change BOTH times.
    x = t.tensor([[1.0, 2.0, 3.0]])
    y = t.tensor([[14.0]])  # true weight = [1, 2, 3]

    snapshots = [model.weight.detach().clone()]
    for _ in range(2):
        loss = ((model(x) - y) ** 2).mean()
        loss.backward()
        fixed.step()
        fixed.zero_grad()
        snapshots.append(model.weight.detach().clone())

    # Both updates must actually move the weights.
    delta_1 = (snapshots[1] - snapshots[0]).abs().sum().item()
    delta_2 = (snapshots[2] - snapshots[1]).abs().sum().item()
    assert delta_1 > 1e-5, f'first step did not update weights: delta={delta_1}'
    assert delta_2 > 1e-5, (
        f'second step did not update weights: delta={delta_2}; '
        f'this is exactly the generator-consumed bug — did you wrap params in list()?'
    )

    # Now demonstrate the bug by running the BUGGY optimizer on a fresh model.
    model2 = t.nn.Linear(3, 1, bias=False)
    with t.no_grad():
        model2.weight.copy_(t.zeros_like(model2.weight))
    buggy = BuggyOptimizer(model2.parameters(), lr=0.1)
    snap2 = [model2.weight.detach().clone()]
    for _ in range(2):
        loss = ((model2(x) - y) ** 2).mean()
        loss.backward()
        buggy.step()
        buggy.zero_grad()
        snap2.append(model2.weight.detach().clone())
    delta_1b = (snap2[1] - snap2[0]).abs().sum().item()
    delta_2b = (snap2[2] - snap2[1]).abs().sum().item()
    assert delta_1b > 1e-5, 'buggy step 1 should still work (generator not yet drained)'
    assert delta_2b < 1e-9, (
        f'buggy step 2 should be a no-op (generator consumed), but moved by {delta_2b}; '
        f'are you sure you are testing the buggy version?'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class BuggyOptimizer:
    # BUG: stores generator as-is
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex1FixedOptimizer:
    def __init__(self, params, lr):
        self.params = list(params)   # <-- the critical fix
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Why this bug is so insidious.** It survives one full step. Loss does decrease on iteration 1. The optimizer 'works' for exactly one batch. Tests that use a single-step smoke check will pass. Then every subsequent step is a silent no-op and the loss curve flatlines.

**PyTorch's built-in optimizers already do this for you.** `torch.optim.SGD.__init__` calls `param_groups = list(params)` internally — that's why you never see this bug with the official API. The trap only matters for hand-rolled optimizers, which is exactly what ARENA chapter 0 part 3 asks you to write.

**General Python lesson.** Any constructor that takes an `Iterable[T]` and intends to iterate it MORE THAN ONCE must materialize it: `list(it)`, `tuple(it)`, or `dict(it)`. The type annotation `Iterable` is the warning sign.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()